[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-org/pypath/blob/main/notebooks/module7/08-fine-tuning.ipynb)

# Fine-Tuning Pre-Trained Transformers
**Module 7 — Lesson 8 | Estimated time: 40 minutes**

> 💡 Enable GPU: Runtime → Change runtime type → GPU (required for this notebook)

## Learning Objectives
By the end of this notebook you will be able to:
- Load and pre-process a real NLP dataset from Hugging Face `datasets`
- Tokenise a dataset using the `map` function
- Use `DataCollatorWithPadding` for efficient dynamic batching
- Fine-tune `AutoModelForSequenceClassification` with the Trainer API
- Set `TrainingArguments` with gradient accumulation
- Compute accuracy and F1 with the `evaluate` library
- Run inference on custom text after fine-tuning

In [ ]:
!pip install -q transformers datasets evaluate accelerate

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
)
import evaluate

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: Training on CPU will be slow. Enable GPU in Runtime settings.')

## 1. Load the SST-2 Dataset

Stanford Sentiment Treebank (SST-2) is a binary sentiment classification benchmark from the GLUE suite. Each sample is a movie review snippet labelled positive (1) or negative (0).

In [ ]:
# Load SST-2 from GLUE benchmark
raw_datasets = load_dataset('glue', 'sst2')
print(raw_datasets)
print('\nTrain sample:')
print(raw_datasets['train'][0])
print('\nLabel mapping: 0=negative, 1=positive')

# Use a smaller subset to keep training under 10 min on Colab GPU
train_data = raw_datasets['train'].select(range(3000))
val_data   = raw_datasets['validation']  # 872 examples
print(f'\nTrain size: {len(train_data)}, Val size: {len(val_data)}')

## 2. Tokenisation with map()

In [ ]:
MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['sentence'],
        truncation=True,
        max_length=128,
    )

tokenized_train = train_data.map(tokenize_fn, batched=True)
tokenized_val   = val_data.map(tokenize_fn,   batched=True)

# Remove original columns not needed by the model
tokenized_train = tokenized_train.remove_columns(['sentence', 'idx'])
tokenized_val   = tokenized_val.remove_columns(['sentence', 'idx'])
tokenized_train = tokenized_train.rename_column('label', 'labels')
tokenized_val   = tokenized_val.rename_column('label', 'labels')
tokenized_train.set_format('torch')
tokenized_val.set_format('torch')

print('Tokenised train features:', tokenized_train.features)
print('Sample:', {k: v[:5] for k, v in tokenized_train[0].items()})

## 3. DataCollator and Model

In [ ]:
# Dynamic padding: pad each batch to the longest sequence in that batch
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Load pre-trained model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: 'NEGATIVE', 1: 'POSITIVE'},
    label2id={'NEGATIVE': 0, 'POSITIVE': 1},
)
print(f'Model loaded: {MODEL_NAME}')
print(f'Total parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## 4. Evaluation Metrics

In [ ]:
accuracy_metric = evaluate.load('accuracy')
f1_metric       = evaluate.load('f1')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)['accuracy']
    f1  = f1_metric.compute(predictions=predictions, references=labels, average='binary')['f1']
    return {'accuracy': acc, 'f1': f1}

print('Metrics: accuracy + F1')

## 5. TrainingArguments and Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir='/tmp/distilbert-sst2',
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    gradient_accumulation_steps=2,      # effective batch = 32*2 = 64
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    lr_scheduler_type='cosine',
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    logging_steps=50,
    report_to='none',                   # disable wandb
    fp16=torch.cuda.is_available(),     # mixed precision on GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print('Trainer ready. Starting fine-tuning...')
train_result = trainer.train()
print('\nTraining complete.')
print(f'Runtime: {train_result.metrics["train_runtime"]:.1f}s')
print(f'Samples/sec: {train_result.metrics["train_samples_per_second"]:.1f}')

## 6. Evaluate on Validation Set

In [ ]:
eval_results = trainer.evaluate()
print('Validation results:')
for k, v in eval_results.items():
    if not k.startswith('eval_runtime'):
        print(f'  {k}: {v:.4f}')

## 7. Training Loss Curve

In [ ]:
log_history = trainer.state.log_history
train_logs  = [l for l in log_history if 'loss' in l and 'eval_loss' not in l]
eval_logs   = [l for l in log_history if 'eval_loss' in l]

if train_logs:
    steps  = [l['step'] for l in train_logs]
    t_loss = [l['loss'] for l in train_logs]
    plt.figure(figsize=(8, 4))
    plt.plot(steps, t_loss, label='Train loss', color='steelblue')
    if eval_logs:
        e_steps = [l['step'] for l in eval_logs]
        e_loss  = [l['eval_loss'] for l in eval_logs]
        e_acc   = [l.get('eval_accuracy', None) for l in eval_logs]
        plt.plot(e_steps, e_loss, 'o--', label='Val loss', color='darkorange')
    plt.xlabel('Training step'); plt.ylabel('Loss')
    plt.title('Fine-Tuning DistilBERT on SST-2')
    plt.legend(); plt.tight_layout(); plt.show()

## 8. Inference on Custom Text

In [ ]:
from transformers import pipeline as hf_pipeline

classifier = hf_pipeline(
    'sentiment-analysis',
    model=model,
    tokenizer=tokenizer,
    device=0 if torch.cuda.is_available() else -1,
)

custom_texts = [
    'This movie had incredible cinematography and a gripping story.',
    'I fell asleep halfway through — a complete waste of time.',
    'An average film: some good moments but largely forgettable.',
    'The acting was superb and the plot kept me on the edge of my seat!',
    'Poorly written script, bad acting, avoid at all costs.',
]

print('Fine-tuned model predictions:')
for text in custom_texts:
    result = classifier(text)[0]
    print(f'  [{result["label"]} {result["score"]:.3f}] {text[:60]}...')

## Practice Exercises

**Exercise 1 — Freeze Layers**
Freeze the bottom 3 transformer layers of DistilBERT (`model.distilbert.transformer.layer[:3]`) and only fine-tune the top layers and classifier head. Compare training speed and final accuracy versus full fine-tuning.

**Exercise 2 — Multi-Class Classification**
Load the `ag_news` dataset (4 categories: world, sports, business, science/tech). Fine-tune DistilBERT for 4-class classification. Set `num_labels=4` and report per-class F1 using `evaluate.load('f1', config_name='macro')`.

**Exercise 3 — Learning Rate Search**
Train the same model with learning rates `[5e-6, 2e-5, 5e-5, 1e-4]` for 1 epoch each. Plot validation accuracy vs learning rate on a log scale and identify the best value.